# Training con Time-Domain features 

Este NoteBook se centra en entrenar clasificadores con las características de dominio temporal de los datos, a diferencia de los otros, que entrenamos con datos crudos. 
El objetivo es comparar clasificadores con FE vs datos crudos, en aspectos como:
- Accuracy
- Latencia
- Tiempo de training
- Robustez al ruido

Aplicaremos cross validation para obtener unos promedios más confiables.

In [1]:
import pandas as pd
import numpy as np
import time
from td_features import td_features_dataset
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)

## Cargamos Datos

In [2]:
df = pd.read_csv("dataset.csv") # dataset completo
y = df.iloc[:, -1].values
df = df.drop(df.columns[-1], axis=1) # eliminamos label

## Pasamos a características del dominio temporal

In [3]:
df = td_features_dataset(df, scaler=StandardScaler())
X = df.values

## Splits

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

## Entrenando Clasificadores 

### SVM

In [5]:
param_grid = [
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    },
    {
        "kernel": ["poly"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto"],
        "degree": [2, 3, 4],
        "coef0": [0.0, 1.0],
    },
]

grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

start = time.time()
grid.fit(X_train, y_train)
end = time.time()
svm_training_time = end - start
best_svm = grid.best_estimator_

print("Mejor estimador:", grid.best_estimator_)
print("Mejores parámetros:", grid.best_params_)
print("Mejor score CV:", grid.best_score_)
print(f"Tiempo de training: {svm_training_time:.2f} s")

Mejor estimador: SVC(C=100, gamma=0.001)
Mejores parámetros: {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
Mejor score CV: 0.6113277735618785
Tiempo de training: 335.83 s


### Random Forest

In [6]:
param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_features':     ['sqrt', 'log2', None],
    'max_depth':        [100, 10, 20],
    'min_samples_leaf': [1, 2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

start = time.time()
grid_rf.fit(X_train, y_train)
end = time.time()
rf_training_time = end - start
best_rf_model = grid_rf.best_estimator_

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"Mejor CV score:     {grid_rf.best_score_:.4f}")
print(f"Test score:         {grid_rf.best_estimator_.score(X_test, y_test):.4f}")
print(f"Tiempo de training: {rf_training_time:.2f} s")

Mejores parámetros: {'max_depth': 100, 'max_features': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Mejor CV score:     0.5436
Test score:         0.5531
Tiempo de training: 292.39 s


### AdaBoost

In [7]:
param_grid = {
    'n_estimators':  [25, 50, 75, 100],
    'learning_rate': [0.01, 0.1, 0.5, 0.9],
    'estimator__max_depth' : [1, 2, 3, 5]
}

grid_ada = GridSearchCV(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

start = time.time()
grid_ada.fit(X_train, y_train)
end = time.time()
ada_training_time = end - start
best_ada_model = grid_ada.best_estimator_

print(f"Mejores parámetros: {grid_ada.best_params_}")
print(f"Mejor CV score:     {grid_ada.best_score_:.4f}")
print(f"Test score:         {grid_ada.best_estimator_.score(X_test, y_test):.4f}")
print(f"Tiempo de training: {ada_training_time:.2f} s")

Mejores parámetros: {'estimator__max_depth': 5, 'learning_rate': 0.1, 'n_estimators': 100}
Mejor CV score:     0.5260
Test score:         0.5486
Tiempo de training: 115.70 s


## Ruido y Latencia

In [8]:
noise = np.random.uniform(-0.3*np.std(X_test), 0.3*np.std(X_test), X_test.shape)
X_noise = X_test + noise

In [9]:
def accuracy_and_latency(model, X_test, y_test):
    latencias = []
    for _ in range(5):
        start = time.time()
        y_pred = model.predict(X_test)
        end = time.time()
        latencias.append(end-start)
    elapsed = np.mean(latencias)
    acc = accuracy_score(y_test, y_pred)
    return [acc, elapsed]

In [14]:
# sin ruido
acc_svm, time_svm = accuracy_and_latency(best_svm, X_test, y_test)
acc_rf, time_rf = accuracy_and_latency(best_rf_model, X_test, y_test)
acc_ada, time_ada = accuracy_and_latency(best_ada_model, X_test, y_test)

# con ruido
acc_svm_n, time_svm_n = accuracy_and_latency(best_svm, X_noise, y_test)
acc_rf_n, time_rf_n = accuracy_and_latency(best_rf_model, X_noise, y_test)
acc_ada_n, time_ada_n = accuracy_and_latency(best_ada_model, X_noise, y_test)

# latencias
lat_svm = (time_svm+time_svm_n) / 2
lat_rf = (time_rf+time_rf_n) / 2
lat_ada = (time_ada+time_ada_n) / 2


pd.DataFrame({
    "Modelo": ["SVM", "Random Forest", "AdaBoost"],
    "Accuracy": [f"{acc_svm:.4f}", f"{acc_rf:.4f}", f"{acc_ada:.4f}"],
    "Accuracy Ruido": [f"{acc_svm_n:.4f}", f"{acc_rf_n:.4f}", f"{acc_ada_n:.4f}"],
    "Latencia (s)": [f"{lat_svm:.4f}", f"{lat_rf:.4f}", f"{lat_ada:.4f}"],
    "Tiempo Training (s)" : [f"{svm_training_time//60:.0f} min {svm_training_time%60:.0f}s",
                            f"{rf_training_time//60:.0f} min {rf_training_time%60:.0f}s", 
                            f"{ada_training_time//60:.0f} min {ada_training_time%60:.0f}s"]
})

,Modelo,Accuracy,Accuracy Ruido,Latencia (s),Tiempo Training (s)
0,SVM,0.6305,0.5699,0.7714,5 min 36s
1,Random Forest,0.5531,0.5476,0.0538,4 min 52s
2,AdaBoost,0.5486,0.5435,0.0263,1 min 56s
